# Домашнее задание к лекции «Продвинутый pandas»

## Задание 1
Для датафрейма `log` из материалов занятия создайте столбец `source_type` по правилам:

* если источник `traffic_source` равен _Yandex_ или _Google_, то в `source_type` ставится `organic`;
* для источников `paid` и `email` из России ставим `ad`;
* для источников `paid` и `email` не из России ставим `other`;
* все остальные варианты берём из `traffic_source` без изменений.

In [30]:
import pandas as pd

In [31]:
log = pd.read_csv('resources/visit_log.csv', sep=';')

log['source_type'] = log['traffic_source']
log.loc[log['traffic_source'].isin(['yandex', 'google']), 'source_type'] = 'organic'
log.loc[(log['traffic_source'].isin(['paid', 'email'])) & (log['region'] == 'Russia'), 'source_type'] = 'ad'
log.loc[(log['traffic_source'].isin(['paid', 'email'])) & (log['region'] != 'Russia'), 'source_type'] = 'other'

print(log[['traffic_source', 'region', 'source_type']].head(10))

  traffic_source  region source_type
0         yandex  Russia     organic
1         direct  Russia      direct
2         yandex  Russia     organic
3         yandex  Russia     organic
4         yandex  Russia     organic
5         yandex  Russia     organic
6           paid  Russia          ad
7         direct  Russia      direct
8         direct   China      direct
9          email  Russia          ad


## Задание 2
В файле `URLs.txt` содержатся URL страниц новостного сайта. Вам нужно отфильтровать его по адресам страниц с текстами новостей. Известно, что шаблон страницы новостей имеет внутри URL конструкцию: /, затем 8 цифр, затем дефис. Выполните действия:

1. Прочитайте содержимое файла с датафрейм.
2. Отфильтруйте страницы с текстом новостей, используя метод str.contains и регулярное выражение в соответствие с заданным шаблоном.

In [32]:
urls = pd.read_csv('resources/URLs.txt', header=None, names=['url'])

news_pattern = r'/\d{8}-'
news_urls = urls[urls['url'].str.contains(news_pattern, na=False)]

news_urls.head()

,url
4,/politics/36188461-s-marta-zhizn-rossiyan-susc...
5,/world/36007585-tramp-pridumal-kak-reshit-ukra...
6,/science/36157853-nasa-sobiraet-ekstrennuyu-pr...
7,/video/36001498-poyavilis-pervye-podrobnosti-g...
8,/world/36007585-tramp-pridumal-kak-reshit-ukra...


## Задание 3
Используйте файл с оценками фильмов `ml-latest-small/ratings.csv`. Посчитайте среднее время жизни пользователей, которые выставили более 100 оценок. Под временем жизни понимается разница между максимальным и минимальным значениями столбца `timestamp` для данного значения `userId`.

In [33]:
ratings = pd.read_csv('resources/ratings.csv')

user_stats = ratings.groupby('userId').agg(
    rating_count=('timestamp', 'count'),
    min_time=('timestamp', 'min'),
    max_time=('timestamp', 'max')
)

active_users = user_stats[user_stats['rating_count'] > 100].copy()
active_users.loc[:, 'lifetime'] = active_users['max_time'] - active_users['min_time']
avg_lifetime = active_users['lifetime'].mean()

print(f"Среднее время жизни: {avg_lifetime:.0f} секунд")

Среднее время жизни: 40080507 секунд


## К домашнему заданию №4
Дана статистика услуг перевозок клиентов компании по типам:
- rzd - железнодорожные перевозки
- auto - автомобильные перевозки
- air - воздушные перевозки
- client_base - адреса клиентов

In [34]:
rzd = pd.DataFrame(
    {
        'client_id': [111, 112, 113, 114, 115],
        'rzd_revenue': [1093, 2810, 10283, 5774, 981]
    }
)

auto = pd.DataFrame(
    {
        'client_id': [113, 114, 115, 116, 117],
        'auto_revenue': [57483, 83, 912, 4834, 98]
    }
)

air = pd.DataFrame(
    {
        'client_id': [115, 116, 117, 118],
        'air_revenue': [81, 4, 13, 173]
    }
)

client_base = pd.DataFrame(
    {
        'client_id': [111, 112, 113, 114, 115, 116, 117, 118],
        'address': ['Комсомольская 4', 'Энтузиастов 8а', 'Левобережная 1а', 'Мира 14', 'ЗЖБИиДК 1',
                    'Строителей 18', 'Панфиловская 33', 'Мастеркова 4']
    }
)

## Задание 4
Дана статистика услуг перевозок клиентов компании по типам (см. файл `Python_13_join.ipynb` в разделе «Материалы для лекции “Продвинутый pandas”» ---- Ноутбуки к лекции «Продвинутый pandas»).
Нужно сформировать две таблицы:

1. таблицу с тремя типами выручки для каждого `client_id` без указания адреса клиента;
2. аналогичную таблицу по типам выручки с указанием адреса клиента.

_Обратите внимание, что в процессе объединения таблиц данные не должны теряться._

In [35]:
result = rzd.merge(auto, on='client_id', how='outer')
result = result.merge(air, on='client_id', how='outer')

print(result.head())

   client_id  rzd_revenue  auto_revenue  air_revenue
0        111       1093.0           NaN          NaN
1        112       2810.0           NaN          NaN
2        113      10283.0       57483.0          NaN
3        114       5774.0          83.0          NaN
4        115        981.0         912.0         81.0


In [36]:
result_with_address = result.merge(client_base, on='client_id', how='left')
print(result_with_address.head())

   client_id  rzd_revenue  auto_revenue  air_revenue          address
0        111       1093.0           NaN          NaN  Комсомольская 4
1        112       2810.0           NaN          NaN   Энтузиастов 8а
2        113      10283.0       57483.0          NaN  Левобережная 1а
3        114       5774.0          83.0          NaN          Мира 14
4        115        981.0         912.0         81.0        ЗЖБИиДК 1
